# Phase 1 - Train the Classifier Head

This is the first step of training. We load MobileNetV2 with its pretrained ImageNet weights and lock all of those layers so they do not change. Then we add a new 2-class output layer on top and train only that new part.

The idea is simple. MobileNetV2 already knows how to look at images and extract useful patterns from its ImageNet training. We do not want to mess with that. We just want to teach the new output layer what empty and occupied parking slots look like.

## Imports

In [ ]:
import os
import gc
import json
import csv
import sys
import shutil
import subprocess
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from tqdm import tqdm

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

import psutil
def log_mem(msg=""):
    gb = 1024**3
    ram = psutil.virtual_memory()
    print(f"[MEM {msg}] RAM {ram.used/gb:.1f}/{ram.total/gb:.1f} GB ({ram.percent}%)")
    if torch.cuda.is_available():
        r = torch.cuda.memory_reserved(0) / gb
        a = torch.cuda.memory_allocated(0) / gb
        print(f"  GPU: allocated {a:.2f} GB, reserved {r:.2f} GB")
    sys.stdout.flush()

## Settings

All the numbers and paths are in one place so they are easy to change.

Change KAGGLE_USERNAME to your Kaggle username.

In [ ]:
DATA_DIR  = "/kaggle/input/datasets/raahad/parking-occupancy-merged"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
VAL_DIR   = os.path.join(DATA_DIR, "val")

# The export folder and dataset name.
KAGGLE_USERNAME  = "kaggle-username"
DATASET_NAME     = "citypulse-phase1-weights"
EXPORT_DIR       = f"/kaggle/working/{DATASET_NAME}"
WEIGHTS_FILENAME = "phase1_weights.pth"

EPOCHS        = 5
BATCH_SIZE    = 128
LEARNING_RATE = 1e-3

# MobileNetV2 was trained with these normalization values so we use the same.
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on:", DEVICE)

## Load the Dataset

The dataset folders are named after the class, so torchvision loads them automatically. The folder named empty becomes class 0 and occupied becomes class 1.

For training we apply some light random changes to the images so the model does not just memorize them. For validation we do nothing except resize and normalize, because we want a fair measure of how well it actually works.

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset   = datasets.ImageFolder(VAL_DIR,   transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=False)

print("Classes:", train_dataset.classes)
print("Train samples:", len(train_dataset))
print("Val samples:  ", len(val_dataset))

## Build the Model

We load MobileNetV2 with the weights it got from training on ImageNet. Then we swap out the last layer. The original last layer outputs 1000 classes. We replace it with a 2-class layer since we only care about empty and occupied.

After that we freeze all the layers except the new one. Freezing means those layers will not update during training. Only the new classifier layer learns in this phase.

In [ ]:
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)

# Replace the 1000-class head with a 2-class head
model.classifier = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(model.last_channel, 2)
)

# Lock everything except the new classifier
for param in model.features.parameters():
    param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} out of {total:,} parameters")

model = model.to(DEVICE)

## Training and Validation Functions

Two simple functions. One runs through the training data and updates the weights. The other runs through the validation data without updating anything, just to measure accuracy.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device, epoch, total_epochs):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(loader, desc=f"Epoch {epoch}/{total_epochs} train", leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        loss_val = loss.item()
        total_loss += loss_val * images.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += images.size(0)
        del images, labels, outputs, loss
        pbar.set_postfix(loss=f"{loss_val:.4f}", acc=f"{correct/total:.4f}")
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device, epoch, total_epochs):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(loader, desc=f"Epoch {epoch}/{total_epochs} val  ", leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss_val = loss.item()
        total_loss += loss_val * images.size(0)
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += images.size(0)
        del images, labels, outputs, loss
        pbar.set_postfix(loss=f"{loss_val:.4f}", acc=f"{correct/total:.4f}")
    return total_loss / total, correct / total

## Run Phase 1 Training

Train for 5 epochs. Only the new classifier layer is updating. The backbone just reads images and passes the features forward without changing.

In [ ]:
EPOCHS_DIR = os.path.join(EXPORT_DIR, 'epochs')
os.makedirs(EPOCHS_DIR, exist_ok=True)
BEST_WEIGHTS = os.path.join(EXPORT_DIR, WEIGHTS_FILENAME)
CSV_PATH = os.path.join(EXPORT_DIR, 'training_log.csv')

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE
)

best_val_acc = 0.0
best_epoch = 0
history = []

log_mem("before training")

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE, epoch, EPOCHS)
    vl_loss, vl_acc = evaluate(model, val_loader, criterion, DEVICE, epoch, EPOCHS)
    print(f"Epoch {epoch}/{EPOCHS}   train loss {tr_loss:.4f}  acc {tr_acc:.4f}   val loss {vl_loss:.4f}  acc {vl_acc:.4f}", end="")

    history.append({
        'epoch': epoch,
        'train_loss': tr_loss, 'train_acc': tr_acc,
        'val_loss': vl_loss, 'val_acc': vl_acc,
    })

    epoch_path = os.path.join(EPOCHS_DIR, f'phase1_epoch_{epoch}.pth')
    torch.save(model.state_dict(), epoch_path)

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        best_epoch = epoch
        torch.save(model.state_dict(), BEST_WEIGHTS)
        print("  <- best saved")
    else:
        print()

    gc.collect()
    torch.cuda.empty_cache()
    log_mem(f"epoch {epoch} done")

with open(CSV_PATH, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['epoch','train_loss','train_acc','val_loss','val_acc'])
    writer.writeheader()
    writer.writerows(history)

print(f"\nBest: epoch {best_epoch} val_acc {best_val_acc:.4f}")
print(f"All epochs saved: {EPOCHS_DIR}")
print(f"Best weights: {BEST_WEIGHTS}")
print(f"CSV log: {CSV_PATH}")